# 00 · Inventário e qualidade dos dados

Ponto de partida da análise: **o que existe em `dados/processed/`, de quando, e o quão completo está**.
Nenhuma conclusão de treino sai daqui — este notebook responde ao que vem antes: quantas linhas, que
período, qual o grão de cada tabela e onde há buraco.

## Como os notebooks se dividem

Um notebook por **domínio de observação**, não por etapa de código. Cada um carrega só as tabelas que
usa, roda sozinho do início ao fim e termina numa seção **Observações** — é ali que a leitura dos
dados fica registrada, em vez de sumir na saída de uma célula.

| Notebook | Pergunta que responde | Tabelas |
|---|---|---|
| `00_overview` | O que existe no dado e dá para confiar nele? | todas |
| `01_wellness` | Como estão sono, HRV e recuperação? | `wellness` |
| `02_activities` | Como o treino se distribui e como a forma evolui? | `activities`, `hr_zones` |
| `03_load_recovery` | A carga de treino explica a recuperação? | `wellness` + `training_load` |

A ordem é de dependência de leitura, não de execução: `00` explica o que existe, `01` e `02` olham
cada fonte no seu próprio recorte, `03` só faz sentido depois que os dois lados são conhecidos.

## Setup

**Nenhum notebook chama `pd.read_csv`.** Toda tabela vem de um `load_*()` de `processing.datasets`,
que já resolve o caminho pelo `config`, converte as datas e cria as colunas de semana com a mesma
âncora. Duas consequências práticas: a mesma tabela chega idêntica em qualquer notebook, e uma
correção de leitura é feita num lugar só em vez de em quatro células espalhadas.

A convenção de nome é a mesma em todos: a variável tem o nome da tabela (`activities`, `wellness`,
`hr_zones`), recortes derivados levam o nome do filtro (`runs`, `base_runs`), e nada leva prefixo
`df_` — todas são DataFrames.

In [ ]:
from processing.datasets import GRAO, load_all, week_origin
from processing.features import auditar_nulos

import matplotlib.pyplot as plt
import pandas as pd

tabelas = load_all()

print(f"{len(tabelas)} tabelas carregadas")
print(f"Semana 1 ancorada em {week_origin().date()} (segunda-feira da atividade mais antiga)")

## 1. Inventário

Uma linha por tabela: tamanho, grão e janela coberta. O **grão** é a informação que mais evita erro
mais adiante — é ele que diz se um `groupby` faz sentido e se um merge é 1:1 ou N:1. `hr_zones` tem
574 linhas para 176 atividades justamente porque o grão é (atividade, zona), não atividade.

In [ ]:
def coluna_data(df):
    return "week_start" if "week_start" in df.columns else "date"


inventario = pd.DataFrame([
    {
        "tabela": nome,
        "linhas": len(df),
        "colunas": df.shape[1],
        "grão": GRAO[nome],
        "de": df[coluna_data(df)].min().date(),
        "até": df[coluna_data(df)].max().date(),
        "semanas": f"{df['semana'].min()}–{df['semana'].max()}",
    }
    for nome, df in tabelas.items()
])

inventario

## 2. Cobertura temporal — as janelas não coincidem

Ponto que condiciona toda análise cruzada: as tabelas cobrem períodos bem diferentes. As atividades
vêm do Strava e vão até 2024; wellness vem do export GDPR do Garmin, que só entrega alguns meses para
trás. Qualquer análise que cruze treino com sono/HRV vive na **interseção** das duas janelas — e é
por isso que ela mora num notebook separado (`03_load_recovery`), com o recorte explícito, em vez de
aparecer no meio de uma análise de treino.

A numeração de `semana` é global (mesma âncora em todas as tabelas), então `semana 100` é a mesma
semana do calendário em qualquer uma delas. É o que permite cruzar por semana sem alinhar datas na mão.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

nomes = list(tabelas)[::-1]  # o primeiro do dicionário fica no topo do gráfico

for i, nome in enumerate(nomes):
    df = tabelas[nome]
    inicio, fim = df[coluna_data(df)].min(), df[coluna_data(df)].max()
    ax.barh(i, (fim - inicio).days, left=inicio, height=0.55, color="#3498db", alpha=0.85)
    ax.text(fim + pd.Timedelta(days=10), i, f"{len(df)} linhas", va="center", fontsize=9, color="gray")

ax.set_yticks(range(len(nomes)), nomes)
ax.set_title("Janela coberta por cada tabela")
ax.grid(axis="x", alpha=0.3)
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

## 3. Auditoria de nulos

Percentual de faltantes por tabela, do pior para o melhor. **Ler sempre contra o grão:** `activities`
é larga de propósito — colunas de natação ficam vazias em 175 de 176 linhas e isso não é defeito, é o
formato. O que merece atenção são as colunas 100% vazias (nunca preenchidas por fonte nenhuma) e as
que estão parcialmente preenchidas dentro do mesmo esporte, porque essas mudam o `n` de qualquer
análise sem avisar.

In [ ]:
for nome, df in tabelas.items():
    nulos = auditar_nulos(df)
    vazias = nulos.loc[nulos["pct"] == 100, "coluna"].tolist()

    print(f"\n=== {nome} — {len(nulos)} de {df.shape[1]} colunas com faltantes ===")
    if not nulos.empty:
        print(nulos.head(5).to_string(index=False))
    if vazias:
        print(f"  sempre vazias ({len(vazias)}): {', '.join(vazias[:10])}")

## 4. De onde vem cada atividade

`source` diz qual export originou a linha, e isso explica boa parte dos nulos de `activities`:
`strava+garmin` traz zonas de FC, RPE, calorias e dinâmica de corrida; `strava` puro não tem nada
disso. Filtros que dependem de campo só-Garmin (`training_effect_label`, por exemplo) reduzem a
amostra silenciosamente — conferir aqui antes evita concluir sobre 46 corridas achando que são 126.

In [ ]:
activities = tabelas["activities"]
runs = activities[activities["sport"] == "Run"]

print(activities["source"].value_counts().to_string())
print()
print(activities["sport"].value_counts().to_string())
print()
print(f"Corridas: {len(runs)}")
print(f"  com FC e pace:            {runs[['avg_heart_rate', 'avg_pace_sec_per_km']].dropna().shape[0]}")
print(f"  com training_effect_label: {runs['training_effect_label'].notna().sum()}  (campo só do Garmin)")

## Observações

Registrar aqui o que a rodada de dados mostrou — o valor deste notebook é o histórico dessas notas,
não os gráficos. Atualizar depois de cada `export_context.py`.

**Estado em 2026-08 (176 atividades, 119 dias de wellness):**

- **`spo2` está 100% vazia** — a coluna existe no CSV mas nenhum registro tem valor. Não usar em
  análise até o relógio passar a gravar.
- **`vo2max` só tem 29% dos dias** (35 de 119): o relógio só recalcula a estimativa depois de certos
  treinos. Para série temporal, preencher para a frente (`ffill`) ou tratar como evento, nunca como
  medida diária.
- **`training_effect_label` existe em 45 das 126 corridas** — é campo Garmin, e 94 das 176 atividades
  vieram só do Strava. Toda análise por tipo de treino roda sobre esse subconjunto.
- **`training_load` vai 11 dias além da última atividade** (até 2026-08-17): é o decaimento de
  CTL/ATL com TSS zero, comportamento esperado do modelo, não dado faltando.
- **Janela útil para cruzar treino × recuperação: 2026-04-08 a 2026-08-04** (119 dias). Fora dela só
  existe o lado do treino.